In [16]:
import psycopg
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

In [ ]:
load_dotenv() #library used to read the env file
DDL = """
CREATE EXTENSION IF NOT EXISTS vector;

DROP TABLE IF EXISTS chunks CASCADE;

CREATE TABLE chunks (
    chunk_id        TEXT PRIMARY KEY,
    text            TEXT NOT NULL,
    embedding       vector(384) NOT NULL,
    jurisdiction    TEXT NOT NULL,
    act             TEXT,
    section_number  TEXT,
    section_title   TEXT,
    part            TEXT,
    division        TEXT,
    schedule        TEXT,
    source_url      TEXT,
    version         TEXT,
    downloaded_date TEXT
);

CREATE INDEX chunks_embedding_idx ON chunks
        USING hnsw (embedding vector_cosine_ops);

CREATE INDEX chunks_jurisdiction_idx ON chunks (jurisdiction);
"""

#This is the code snippet to run a DDL file
with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)
    conn.commit()

print("Table Created")

Table Created


In [ ]:
#Checking if the file exists or not
with psycopg.connect(os.getenv("DATABASE_URL")) as conn:
    with conn.cursor() as cur:
        cur.execute("""
                    SELECT column_name,data_type
                    FROM information_schema.columns
                    WHERE table_name = 'chunks'
                    ORDER BY ordinal_position;
                    """)
        for col, dtype in cur.fetchall():
            print(f" {col:20s} {dtype}")

 chunk_id             text
 text                 text
 embedding            USER-DEFINED
 jurisdiction         text
 act                  text
 section_number       text
 section_title        text
 part                 text
 division             text
 schedule             text
 source_url           text
 version              text
 downloaded_date      text


In [17]:
#Embedding models
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"Output dim: {model.get_sentence_embedding_dimension()}")

sample = model.encode("How much notice for rent increase?")
print(f"Sample shape: {sample.shape}")

Output dim: 384
Sample shape: (384,)
